# Spatial enrichment maps for the significant ASD modules

A background-corrected cortical expression map for each module retained by the stepwise
conditional analysis, built with exactly the method used for the ASD and topic gene sets
in notebooks 04_01–03_02: AHBA DK region x gene matrix, brain-expressed background, per-gene
z-scoring across regions, and `z_vs_bg` against 10,000 random size-matched gene sets.

Two outputs: one surface map per module, and a pairwise comparison asking whether the
module maps resemble each other spatially beyond what spatial autocorrelation alone
produces.

> **Kernel:** the `spin` conda env (`Python (spin)`). **Run from this directory.**
> The module gene lists come from `outputs/analysis/02_module_identification/02_00_sig-module-genes.csv`,
> exported from the network object in R. The module count is taken from that file rather than hardcoded.

## 0) Setup

In [ ]:
import os, re, hashlib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import spatial_helpers as sh
warnings.filterwarnings("ignore")

PROJ = os.path.expanduser("~/Documents/Postdoc/projects/ASD-rarevar-annot")
RESULTS  = f"{PROJ}/outputs/analysis/04_brain_spatial_enrichment"
CACHE    = f"{RESULTS}/spin_cache"
FIG_SUPP = f"{PROJ}/outputs/figures/supplement"
FIG_MAPS = f"{FIG_SUPP}/04_05_module_maps"
for d in (RESULTS, FIG_MAPS, CACHE):
    os.makedirs(d, exist_ok=True)

plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42,
                     "font.family": "sans-serif",
                     "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"]})
TCOL = sh.TOPIC_COLORS

expr, info = sh.load_ahba()
background = sh.brain_expressed_background(expr)
Z = sh.zscore_expression(expr, background)
dk = sh.dk_region_order()

mods = pd.read_csv(f"{PROJ}/outputs/analysis/02_module_identification/02_00_sig-module-genes.csv")
module_info = {m: {"genes": g.gene_name.tolist(), "label": g.module_label.iloc[0],
                   "topic": g.topic.iloc[0]} for m, g in mods.groupby("module")}
print(f"significant modules: {len(module_info)} | background genes: {len(background)}")
print(mods.groupby("topic").module.nunique().to_string())

## 1) Per-module enrichment and surface maps

The enrichment is computed once per module and reused by §2, since 10,000 resamples per
module is the expensive step. Each map is scaled to its **own** max |Z| — the modules
differ substantially in how strongly patterned they are, and a shared scale would flatten
the weaker ones — so the colour limits are not comparable across panels. The title is the
module's hub/TF label, coloured by its GO topic.

In [ ]:
module_enr = {}
for module, d in module_info.items():
    module_enr[d["label"]] = (sh.regional_enrichment(Z, info, d["genes"], n_perm=10000, seed=0),
                              d["topic"])
    print("enriched", d["label"], flush=True)

In [ ]:
# Module maps use a white medial wall (the main-text maps in notebooks 04_01-04_02 use the
# anatomical tan), which reads better across a dense grid of small panels.
for label, (enr, topic) in module_enr.items():
    cmap = sh.diverging_cmap(); cmap.set_over("#FFFFFF")
    z_by_region = dict(zip(enr.region, enr.z_vs_bg))
    vmax = sh.cortical_vmax(enr)
    fig = sh.plot_dk_surface(z_by_region, cmap, vmax, figsize=(7.09, 4),
                             title=label, title_color=TCOL[topic])
    safe = re.sub(r"[^A-Za-z0-9]+", "_", label).strip("_")
    fig.savefig(f"{FIG_MAPS}/{topic}_{safe}.png", dpi=300, bbox_inches="tight")
    plt.show()

## 2) Do the module maps resemble each other?

Observed pairwise Spearman correlation between the module `z_vs_bg` maps across the 34
cortical regions, then a spin test (N=1,000, Cornblath parcel-spin at 41k) asking whether
each pair's correlation exceeds what spatial autocorrelation alone produces. The null for
a pair pools both directions — spinning A against B and B against A — so it is symmetric.
BH-FDR is applied across all pairs.

In [ ]:
module_z = {label: np.array([dict(zip(enr.region, enr.z_vs_bg)).get(r, np.nan) for r in dk])
            for label, (enr, _) in module_enr.items()}
topic_of = {label: topic for label, (_, topic) in module_enr.items()}
order = sorted(module_z, key=lambda L: (["GR", "SYN", "MORPH"].index(topic_of[L]), L))

corr = pd.DataFrame({L: module_z[L] for L in order}).corr(method="spearman")
corr.to_csv(f"{RESULTS}/04_05_module_pairwise_corr.csv")

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=90, fontsize=6)
ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=6)
for i, L in enumerate(order):                       # tick labels coloured by GO topic
    ax.get_xticklabels()[i].set_color(TCOL[topic_of[L]])
    ax.get_yticklabels()[i].set_color(TCOL[topic_of[L]])
cb = fig.colorbar(im, fraction=0.045, pad=0.04)
cb.set_label("Spearman ρ", fontsize=7); cb.ax.tick_params(labelsize=6)
ax.set_title(f"Pairwise module enrichment-map correlation, {len(order)} modules "
             "(ordered by topic)", fontsize=8)
fig.tight_layout()
fig.savefig(f"{FIG_SUPP}/04_05_module_pairwise_corr.pdf", bbox_inches="tight")
fig.savefig(f"{FIG_SUPP}/04_05_module_pairwise_corr.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Spinning each module map takes ~45 s (cornblath at 41k), so the first run is ~20-25 min
# for the full set. Each spun map is cached under outputs/spin_cache keyed by a hash of the
# map values, so re-runs are instant and a changed map invalidates its own cache entry.
from statsmodels.stats.multitest import multipletests
from run_spin_test import resample_parc_to_density
from neuromaps.images import annot_to_gifti
from neuromaps.datasets import fetch_atlas
from neuromaps.nulls import cornblath
from neuromaps.nulls.spins import get_parcel_centroids, gen_spinsamples

N_SPIN, DENSITY, SEED = 1000, "41k", 0
parc = resample_parc_to_density(annot_to_gifti((sh.ANNOT_LH, sh.ANNOT_RH)), DENSITY)

spins_path = f"{CACHE}/spins_{DENSITY}_n{N_SPIN}_seed{SEED}.npy"
if os.path.exists(spins_path):
    spins = np.load(spins_path)
else:
    coords, hemiid = get_parcel_centroids(fetch_atlas("fsaverage", DENSITY)["sphere"],
                                          method="surface")
    spins = gen_spinsamples(coords, hemiid, n_rotate=N_SPIN, seed=SEED, verbose=1)
    np.save(spins_path, spins)

spun = {}
for L in module_z:
    data68 = np.concatenate([module_z[L], np.full(34, np.nan)])      # RH has no data
    h = hashlib.md5(np.ascontiguousarray(data68, np.float64).tobytes()).hexdigest()[:10]
    path = f"{CACHE}/modulespun_{DENSITY}_n{N_SPIN}_seed{SEED}_{h}.npy"
    if os.path.exists(path):
        spun[L] = np.load(path)
    else:
        spun[L] = cornblath(data=data68, atlas="fsaverage", density=DENSITY,
                            parcellation=parc, n_perm=N_SPIN, spins=spins)[:34]
        np.save(path, spun[L])
    print("spun", L, flush=True)

In [ ]:
labels = list(module_z)
rows = []
for a in range(len(labels)):
    for b in range(a + 1, len(labels)):
        A, B = labels[a], labels[b]
        obs = sh.masked_spearman(module_z[A], module_z[B])
        null = np.concatenate([[sh.masked_spearman(spun[A][:, i], module_z[B]) for i in range(N_SPIN)],
                               [sh.masked_spearman(spun[B][:, i], module_z[A]) for i in range(N_SPIN)]])
        rows.append({"module_A": A, "topic_A": topic_of[A],
                     "module_B": B, "topic_B": topic_of[B],
                     "rho": obs, "spin_p": sh.emp_p(obs, null)})
pair_res = pd.DataFrame(rows)
pair_res["fdr"] = multipletests(pair_res.spin_p, method="fdr_bh")[1]
pair_res.sort_values("spin_p").to_csv(f"{RESULTS}/04_05_module_pairwise_spin.csv", index=False)

print(f"{(pair_res.fdr < 0.05).sum()}/{len(pair_res)} pairs significant (FDR < 0.05)")
pair_res.sort_values("spin_p").head(15)